RESNET

In [70]:
import torch
import torchvision.models as models
import onnx
import onnxruntime as ort
import time
import numpy as np

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [71]:
# import json

# with open('class.json', 'r') as f:
#     class_labels = json.load(f)

In [72]:
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)  
resnet50.eval()  


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [73]:
DOG="./images/dog.jpg"
CAR="./images/car.jpg"

In [74]:
from torchvision import transforms
from PIL import Image

image = Image.open(DOG) 

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.ToTensor(),
])

input_tensor = transform(image).unsqueeze(0)  


In [75]:
MODEL_DIR='./models/resnet50.onnx'

In [76]:


# Save the PyTorch model
torch.save(resnet50.state_dict(), './models/resnet50.pth')

# Load the PyTorch model
resnet50_loaded = models.resnet50().to(device)
resnet50_loaded.load_state_dict(torch.load('./models/resnet50.pth'))
resnet50_loaded.eval()

# Measure inference time for PyTorch model
start_time = time.time()
with torch.no_grad():
    output = resnet50_loaded(input_tensor.to(device))
pytorch_inference_time = time.time() - start_time


In [77]:
# Save the ONNX model
torch.onnx.export(
    model=resnet50, 
    args=input_tensor, 
    f=MODEL_DIR,
    input_names=["input"],  
    output_names=["output"]
)

# Load the ONNX model into ONNX Runtime
onnx_session = ort.InferenceSession(MODEL_DIR)

# Measure inference time for ONNX model
start_time = time.time()
inputs = {"input": input_tensor.numpy()}
outputs = onnx_session.run(None, inputs)
onnx_inference_time = time.time() - start_time

In [78]:

print(f"PyTorch Inference Time: {pytorch_inference_time:.6f} seconds")
print(f"ONNX Inference Time: {onnx_inference_time:.6f} seconds")

PyTorch Inference Time: 0.013433 seconds
ONNX Inference Time: 0.053912 seconds


In [79]:
#Calculate the difference percentage

diff = np.abs(pytorch_inference_time-onnx_inference_time)
percentage_diff = (diff / pytorch_inference_time) * 100
print(f"Percentage Difference: {percentage_diff:.6f}%")

Percentage Difference: 301.327559%


CNN

In [80]:
import onnx_graphsurgeon as gs 

# Load the ONNX model

CNN= "./models/CNN.onnx"
IMG = "./images/0.jpg"

transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
])

graph= gs.import_onnx(onnx.load(CNN))
print(graph)


Graph main_graph (Opset 17)
Local Functions: []
Inputs: [Variable (input): (shape=['batch_size', 3, 64, 64], dtype=float32)]
Nodes: /conv_block_1/conv_block_1.0/Conv (Conv)
	Inputs: [
		Variable (input): (shape=['batch_size', 3, 64, 64], dtype=float32)
		Constant (conv_block_1.0.weight): (shape=[10, 3, 3, 3], dtype=float32)
		Constant (conv_block_1.0.bias): (shape=[10], dtype=float32)
	]
	Outputs: [
		Variable (/conv_block_1/conv_block_1.0/Conv_output_0): (shape=None, dtype=None)
	]
Attributes: OrderedDict({'dilations': [1, 1], 'group': 1, 'kernel_shape': [3, 3], 'pads': [1, 1, 1, 1], 'strides': [1, 1]})
/conv_block_1/conv_block_1.1/Relu (Relu)
	Inputs: [
		Variable (/conv_block_1/conv_block_1.0/Conv_output_0): (shape=None, dtype=None)
	]
	Outputs: [
		Variable (/conv_block_1/conv_block_1.1/Relu_output_0): (shape=None, dtype=None)
	]
/conv_block_1/conv_block_1.2/Conv (Conv)
	Inputs: [
		Variable (/conv_block_1/conv_block_1.1/Relu_output_0): (shape=None, dtype=None)
		Constant (conv_blo

In [81]:
graph.inputs

[Variable (input): (shape=['batch_size', 3, 64, 64], dtype=float32)]

In [82]:
for node in graph.nodes:
    print(node.outputs)
 

[Variable (/conv_block_1/conv_block_1.0/Conv_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_1/conv_block_1.1/Relu_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_1/conv_block_1.2/Conv_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_1/conv_block_1.3/Relu_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_1/conv_block_1.4/MaxPool_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_2/conv_block_2.0/Conv_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_2/conv_block_2.1/Relu_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_2/conv_block_2.2/Conv_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_2/conv_block_2.3/Relu_output_0): (shape=None, dtype=None)]
[Variable (/conv_block_2/conv_block_2.4/MaxPool_output_0): (shape=None, dtype=None)]
[Variable (/classifier/classifier.0/Flatten_output_0): (shape=None, dtype=None)]
[Variable (output): (shape=['batch_size', 10], dtype=float32)]


In [83]:
from onnx import helper

In [ ]:
import onnx
from onnx import helper

intermediate_layer1 = helper.make_tensor_value_info(
    name='/conv_block_1/conv_block_1.0/Conv_output_0',  # Name in the ONNX graph
    elem_type=onnx.TensorProto.FLOAT,
     
    shape=(10,64,64)  # Shape of the intermediate tensor
)

graph.outputs.append(intermediate_layer1)

# getting the second one
intermediate_layer2 = helper.make_tensor_value_info(
    name='/conv_block_1/conv_block_1.1/Relu_output_0',  # Name in the ONNX graph
    elem_type=onnx.TensorProto.FLOAT, 
    shape=(10,64,64)  # Shape of the intermediate tensor
)

graph.outputs.append(intermediate_layer2)

print(graph.outputs)

mymodel = gs.export_onnx(graph)  # Export the graph to an ONNX model
# onnx.save(mymodel, "./models/test_export.onnx") 

[Variable (output): (shape=['batch_size', 10], dtype=float32), name: "/conv_block_1/conv_block_1.0/Conv_output_0"
type {
  tensor_type {
    elem_type: 1
    shape {
      dim {
        dim_value: 10
      }
      dim {
        dim_value: 64
      }
      dim {
        dim_value: 64
      }
    }
  }
}
, name: "/conv_block_1/conv_block_1.1/Relu_output_0"
type {
  tensor_type {
    elem_type: 1
    shape {
      dim {
        dim_value: 10
      }
      dim {
        dim_value: 64
      }
      dim {
        dim_value: 64
      }
    }
  }
}
, name: "/conv_block_1/conv_block_1.0/Conv_output_0"
type {
  tensor_type {
    elem_type: 1
    shape {
      dim {
        dim_value: 10
      }
      dim {
        dim_value: 64
      }
      dim {
        dim_value: 64
      }
    }
  }
}
, name: "/conv_block_1/conv_block_1.1/Relu_output_0"
type {
  tensor_type {
    elem_type: 1
    shape {
      dim {
        dim_value: 10
      }
      dim {
        dim_value: 64
      }
      dim {
        d

AttributeError: dtype

In [85]:

onnx_session = ort.InferenceSession(CNN)
image= Image.open(IMG)



start_time = time.time()
inputs = {"input": input_tensor.numpy()}
outputs = onnx_session.run(None, inputs)
onnx_inference_time = time.time() - start_time

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: input for the following indices
 index: 2 Got: 256 Expected: 64
 index: 3 Got: 383 Expected: 64
 Please fix either the inputs/outputs or the model.